# Notebook 61 — Full Physical Parameter Mapping (Ω, Δ, V, γ, γφ) → b

Extend Notebook 60 by mapping *actual physical parameters* (not just ratios)
into universality exponent b.

Goal:
- simulate Lindblad-inspired dynamics with explicit parameters
- explore regimes: drive-dominated, interaction-dominated, dissipative
- map full parameter space → universality class


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit


In [ ]:
x = np.linspace(1e-3,1,400)

def gamma_eff(x, Omega, Delta, V, gamma, gamma_phi):
    # baseline from drive-detuning balance
    base = 1.5 + 0.5*(Omega/(Omega+abs(Delta)+1e-6))

    # dephasing contribution
    deph = gamma_phi * np.exp(-((x-0.3)/0.2)**2)

    # interaction contribution
    interaction = V * np.sin(2*np.pi*x)

    # spontaneous decay
    decay = gamma

    return np.clip(base + deph + interaction + decay, 0.2, None)

def build_S(g):
    dx=x[1]-x[0]
    return np.exp(-np.cumsum(g)*dx)

def stretched(x,a,b): return np.exp(-a*x**b)

def fit_b(S):
    popt,_ = curve_fit(stretched,x,S,p0=[1,1])
    return popt[1]


## Sweep Ω vs V (fixed γ, γφ, Δ)

In [ ]:
Omegas = np.linspace(0.5,2.0,15)
Vs = np.linspace(0.0,1.0,15)

gamma = 0.3
gamma_phi = 0.5
Delta = 0.2

b_map = np.zeros((len(Omegas), len(Vs)))

for i,O in enumerate(Omegas):
    for j,Vv in enumerate(Vs):
        G = gamma_eff(x,O,Delta,Vv,gamma,gamma_phi)
        S = build_S(G)
        b_map[i,j] = fit_b(S)


In [ ]:
plt.imshow(b_map, origin='lower', aspect='auto',
           extent=[Vs[0],Vs[-1],Omegas[0],Omegas[-1]])
plt.colorbar(label='b')
plt.xlabel('V (interaction)')
plt.ylabel('Ω (drive)')
plt.title('Universality map: Ω vs V')
plt.show()


## Sweep γ vs γφ (fixed Ω, V, Δ)

In [ ]:
gammas = np.linspace(0.0,1.0,15)
gamma_phis = np.linspace(0.0,1.0,15)

Omega = 1.0
V = 0.5
Delta = 0.2

b_map2 = np.zeros((len(gammas), len(gamma_phis)))

for i,g in enumerate(gammas):
    for j,gp in enumerate(gamma_phis):
        G = gamma_eff(x,Omega,Delta,V,g,gp)
        S = build_S(G)
        b_map2[i,j] = fit_b(S)


In [ ]:
plt.imshow(b_map2, origin='lower', aspect='auto',
           extent=[gamma_phis[0],gamma_phis[-1],gammas[0],gammas[-1]])
plt.colorbar(label='b')
plt.xlabel('γφ (dephasing)')
plt.ylabel('γ (decay)')
plt.title('Universality map: γ vs γφ')
plt.show()


## Interpretation

- Ω vs V map: competition between drive and interactions
- γ vs γφ map: competition between decay and dephasing

Universality exponent b becomes a function of full physical parameter space.
